### Imports

In [13]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import datasets
import numpy
import os
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification,  Trainer, TrainingArguments, DataCollatorWithPadding

### Load and tokenize data

In [14]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [15]:
values = [ "Self-direction: thought", "Self-direction: action", "Stimulation",  "Hedonism", "Achievement", "Power: dominance", "Power: resources", "Face", "Security: personal", "Security: societal", "Tradition", "Conformity: rules", "Conformity: interpersonal", "Humility", "Benevolence: caring", "Benevolence: dependability", "Universalism: concern", "Universalism: nature", "Universalism: tolerance" ]
labels = sum([[value + " attained", value + " constrained"] for value in values], [])

In [16]:
num_labels = len(labels)
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=num_labels)  

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
def load_dataset(directory, tokenizer, load_labels=True):
    sentences_file_path = os.path.join(directory, "sentences.tsv")
    labels_file_path = os.path.join(directory, "labels.tsv")
    
    data_frame = pd.read_csv(sentences_file_path, encoding="utf-8", sep="\t", header=0)
    encoded_sentences = tokenizer(data_frame["Text"].to_list(), truncation=True)

    if load_labels and os.path.isfile(labels_file_path):
        labels_frame = pd.read_csv(labels_file_path, encoding="utf-8", sep="\t", header=0)
        labels_frame = pd.merge(data_frame, labels_frame, on=["Text-ID", "Sentence-ID"])
        labels_matrix = numpy.zeros((labels_frame.shape[0], len(labels)))
        for idx, label in enumerate(labels):
            if label in labels_frame.columns:
                labels_matrix[:, idx] = (labels_frame[label] >= 0.5).astype(int)
        encoded_sentences["labels"] = labels_matrix.tolist()

    encoded_sentences = datasets.Dataset.from_dict(encoded_sentences)
    
    return encoded_sentences, data_frame["Text-ID"].to_list(), data_frame["Sentence-ID"].to_list()

In [ ]:
directory_test="datasets/valueeval24/test-english"
directory_train="datasets/valueeval24/training-english"
directory_validation="datasets/valueeval24/validation-english"

encoded_sentences_test, text_ids_test, sentence_ids_test = load_dataset(directory_test, tokenizer)
encoded_sentences_train, text_ids_train, sentence_ids_train = load_dataset(directory_train, tokenizer)
encoded_sentences_validation, text_ids_validation, sentence_ids_validation = load_dataset(directory_validation, tokenizer)

### Training parameters

In [19]:
def compute_metrics(pred):
    labels = pred.label_ids
    # Prendre les probabilités prédictives et les convertir en labels binaires avec un seuil de 0.5
    preds = (pred.predictions >= 0.5).astype(int)
    
    # Calcul des métriques pour un problème multi-label
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='samples',zero_division=0)
    accuracy = accuracy_score(labels, preds)
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [20]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,             
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-3,
    evaluation_strategy="epoch",
)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_sentences_train,
    eval_dataset=encoded_sentences_test, 
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [27]:
trainer.train()

  0%|          | 0/33570 [00:00<?, ?it/s]

{'loss': 0.1666, 'grad_norm': 0.3195207118988037, 'learning_rate': 0.0019702114983616323, 'epoch': 0.04}
{'loss': 0.099, 'grad_norm': 0.33561328053474426, 'learning_rate': 0.001940422996723265, 'epoch': 0.09}
{'loss': 0.1003, 'grad_norm': 0.3534465432167053, 'learning_rate': 0.0019106344950848974, 'epoch': 0.13}
{'loss': 0.0946, 'grad_norm': 0.3911263942718506, 'learning_rate': 0.0018808459934465296, 'epoch': 0.18}
{'loss': 0.0975, 'grad_norm': 0.3955483138561249, 'learning_rate': 0.0018510574918081623, 'epoch': 0.22}
{'loss': 0.0985, 'grad_norm': 0.30012309551239014, 'learning_rate': 0.0018212689901697945, 'epoch': 0.27}
{'loss': 0.1008, 'grad_norm': 0.2910669445991516, 'learning_rate': 0.001791480488531427, 'epoch': 0.31}
{'loss': 0.1012, 'grad_norm': 0.2970485985279083, 'learning_rate': 0.0017616919868930592, 'epoch': 0.36}
{'loss': 0.0983, 'grad_norm': 0.18077345192432404, 'learning_rate': 0.0017319034852546918, 'epoch': 0.4}
{'loss': 0.1013, 'grad_norm': 0.41692087054252625, 'lear

  0%|          | 0/3643 [00:00<?, ?it/s]

{'eval_loss': 0.09758833795785904, 'eval_accuracy': 0.49193493033152585, 'eval_f1': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_runtime': 1111.59, 'eval_samples_per_second': 13.106, 'eval_steps_per_second': 3.277, 'epoch': 1.0}
{'loss': 0.0958, 'grad_norm': 0.17456337809562683, 'learning_rate': 0.0013148644623175454, 'epoch': 1.03}


KeyboardInterrupt: 

In [ ]:
test_metrics = trainer.evaluate(encoded_sentences_validation)
print("Évaluation sur le jeu de test:", test_metrics)